In [12]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

MODEL = "google_genai:gemini-3.1-flash-lite"
model = init_chat_model(MODEL)

In [13]:
from typing import Literal
from pydantic import BaseModel, Field

class Triage(BaseModel):
    category: Literal["billing", "technical", "account", "general"] = Field(description="The main topic of the customer's message.")
    urgency: Literal["low", "mediam", "high"] = Field(description="How quickly this needs a response")
    sentiment: Literal["negative", "neutral", "positive"] = Field(description="The customers mood in the message")
    need_human: bool = Field(description="True if this must be escalated to a human — refunds, billing disputes, cancellations, legal/privacy issues, or an angry customer.")
    summary: str = Field(description="one line summary what the customer wants")

In [14]:
import yaml
from pathlib import Path

_PROMPTS = yaml.safe_load((Path.cwd() / "prompt.yml").read_text(encoding="utf-8"))
TRIAGE_SYSTEM = _PROMPTS["triage_system"]
REPLY_SYSTEM = _PROMPTS["reply_system"] 

print("Prompts loaded successfully!")

Prompts loaded successfully!


In [15]:
IN_PRICE_PER_MTOK = 0.25     
OUT_PRICE_PER_MTOK = 1.50    
_totals = {"input": 0, "output": 0}

def track(usage_metadata) -> None:
    if usage_metadata:
        _totals["input"] += usage_metadata.get("input_tokens", 0)
        _totals["output"] += usage_metadata.get("output_tokens", 0)

def usage_report() -> str:
    cost = (_totals["input"] / 1e6 * IN_PRICE_PER_MTOK
            + _totals["output"] / 1e6 * OUT_PRICE_PER_MTOK)
    return f"{_totals['input']} in + {_totals['output']} out tokens = ~${cost:.10f}"

In [16]:
def triage(message: str) -> Triage:
    # Bind the Pydantic schema and request raw envelope metadata
    triage_model = model.with_structured_output(Triage, include_raw=True)
    response = triage_model.invoke([
        {"role": "system", "content": TRIAGE_SYSTEM},
        {"role": "user", "content": message}
    ])
    track(response["raw"].usage_metadata)      
    return response["parsed"]

In [17]:
def lookup_account(customer_email: str) -> str:
    """Look up customer account details and current billing status using their email address."""
    if "charged twice" in customer_email: 
        return "Account status: Active. Issue detected: System error caused duplicate billing on May 1st."
    return "Account status: Active. No billing issues."

def get_knowledge_base_article(topic: str) -> str:
    """Fetch help articles for technical issues or account changes."""
    if "email" in topic.lower():
        return "Article: To change your email address, navigate to Settings > Profile > Update Email."
    elif "crash" in topic.lower():
        return "Article: If the app crashes on upload, ensure the file is under 50MB and clear the app cache."
    return "No specific article found."
northstar_tools = [lookup_account, get_knowledge_base_article]

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    tools=northstar_tools,
    system_prompt=REPLY_SYSTEM
)

def draft_agent_reply(message: str, t: Triage) -> str:
    """Uses the agent to draft a reply, utilizing tools if necessary."""
    user_payload = f"Customer message:\n{message}\n\nTriage context: {t.model_dump()}"
    
    result = agent.invoke({
        "messages": [{"role": "user", "content": user_payload}]
    })
    
    
    final_message = result["messages"][-1]
    if hasattr(final_message, "usage_metadata") and final_message.usage_metadata:
        track(final_message.usage_metadata)
        
    return final_message.text

In [19]:
def main() -> None:
    samples = [
        "I've been charged twice for May and nobody has replied. This is ridiculous.",
        "How do I change the email address on my account?",
        "Your app keeps crashing when I upload a file.",
        "Who are you?",
        "What can you do for me",
        "Can I directly contact with the human support",
    ]
    for s in samples:
        t = triage(s)
        print("-" * 70)
        print("MESSAGE :", s)
        print("TRIAGE  :", t.model_dump())
        print("ESCALATE:", "yes -> human" if t.need_human else "no")
        print("DRAFT   :", draft_agent_reply(s, t))

    print("=" * 70)
    print("TOKENS  :", usage_report())

main()

----------------------------------------------------------------------
MESSAGE : I've been charged twice for May and nobody has replied. This is ridiculous.
TRIAGE  : {'category': 'billing', 'urgency': 'high', 'sentiment': 'negative', 'need_human': True, 'summary': 'Customer is reporting a double charge for May and expressing frustration over lack of support response.'}
ESCALATE: yes -> human
DRAFT   : I am very sorry for the frustration caused by this billing issue and the delay in our response. A specialized teammate is currently reviewing your account details to investigate the duplicate charge and will follow up with you shortly to resolve this.
----------------------------------------------------------------------
MESSAGE : How do I change the email address on my account?
TRIAGE  : {'category': 'account', 'urgency': 'low', 'sentiment': 'neutral', 'need_human': False, 'summary': 'Customer is asking for instructions on how to update the email address associated with their account.'}